# Inspect GRPO Training Dataset

This notebook loads and visualizes samples from the GRPO training dataset (the same format that goes into `GRPOTrainer`). Use it to:

- **Load** a saved dataset from `outputs/overcooked_grpo/artifacts/datasets/` (or a custom path)
- **View** prompts (system + user messages), completions (XML), rewards, and conversation history
- **Verify** that the data matches what you expect before or after training runs

## Quick Reference: Trajectories vs Dataset

**Trajectory** = One complete episode:
- Contains all steps from start to finish
- Each step has: prompts, completions, actions, rewards for both agents
- Example: Episode with 4 steps → 1 trajectory

**Dataset** = Flat list of training examples:
- Each example = one agent at one step
- Formula: `num_examples = num_episodes × num_steps × 2_agents`
- Example: 10 episodes × 4 steps × 2 agents = 80 examples

**Conversion**: Each step in each trajectory becomes 2 dataset examples (one per agent)

See `docs/trajectories_and_dataset_explained.md` for detailed explanation.

## 1. Choose dataset path

Set `DATASET_DIR` to a saved dataset folder (e.g. `outputs/overcooked_grpo/artifacts/datasets/dataset_iter0000_YYYYMMDD_HHMMSS`).  
If `None`, the latest dataset under `outputs/overcooked_grpo/artifacts/datasets/` is used.

In [19]:
import sys
from pathlib import Path

DATASET_DIR = Path("../../../outputs/overcooked_grpo/artifacts/datasets/dataset_iter0006_20260118_070515")

if DATASET_DIR.exists():
    print("Dataset exist!")
else:
    print(f"Dataset not found!")

Dataset exist!


In [20]:
from llm_trainer import load_dataset

ds = load_dataset(DATASET_DIR)
print(f"Examples: {len(ds)}")
print(f"Columns: {list(ds.features.keys())}")
print(f"\nReward: min={min(ds['reward']):.2f}, max={max(ds['reward']):.2f}, mean={sum(ds['reward'])/len(ds):.2f}")

2026-01-18 09:15:51.735 | INFO     | llm_trainer.artifacts:load_dataset:164 - Loading dataset from ../../../outputs/overcooked_grpo/artifacts/datasets/dataset_iter0006_20260118_070515
2026-01-18 09:15:51.809 | SUCCESS  | llm_trainer.artifacts:load_dataset:167 - Loaded dataset with 1200 examples


Examples: 1200
Columns: ['prompt', 'completion', 'reward', 'agent_id', 'episode', 'step', 'conversation_history']

Reward: min=170.08, max=197.99, mean=181.09


## 2. View samples

Each example has:
- **prompt**: chat messages (system + user) that form the input to the model
- **completion**: the model’s raw response (XML with `<communication>` and `<plan>`)
- **reward**, **agent_id**, **episode**, **step**, **conversation_history**

In [21]:
from IPython.display import display, Markdown, HTML

def show_sample(ex, idx: int):
    md = [f"### Sample {idx} — `{ex['agent_id']}` | episode {ex['episode']} step {ex['step']} | reward **{ex['reward']:.2f}**\n"]
    # Prompt
    md.append("#### Prompt (chat messages)\n")
    for m in ex["prompt"]:
        role = m.get("role", "?")
        raw = m.get("content") or m.get("text") or ""
        content = raw[:1000] + (" …" if len(raw) > 1000 else "")
        md.append(f"- **{role}**: {content}\n")
    # Completion
    comp = ex["completion"]
    md.append(f"#### Completion (raw)\n```\n{comp}\n```\n")
    # Conversation history
    if ex.get("conversation_history"):
        md.append("#### Conversation history\n")
        for h in ex["conversation_history"][-5:]:
            md.append(f"- [{h['from']} → {h['to']}]: {h['message'][:200]}{'…' if len(h['message'])>200 else ''}\n")
    display(Markdown("\n".join(md)))

NUM_SAMPLES = 3
for i in range(min(NUM_SAMPLES, len(ds))):
    show_sample(ds[i], i + 1)

### Sample 1 — `agent_0` | episode 0 step 0 | reward **175.63**

#### Prompt (chat messages)

- **system**: You are an agent in a cooperative cooking game called Overcooked. You work with another agent to cook dishes and deliver them to the goal.

Your task:
1. Observe your local environment
2. Communicate with the other agent about your plans and needs
3. Generate an action plan for what you will do next

Available actions:
- cook_dish: Cook a dish in a pot with specific ingredients
- pickup_ingredient: Pick up an ingredient from a pile
- pickup_plate: Pick up a plate from a plate pile
- deliver_dish: Deliver a completed dish to the goal
- wait: Wait for the other agent or for cooking to complete
- move_to: Move to a specific location

Respond in the following XML format:
<communication>
  <to_agent>agent_X</to_agent>
  <message>Your message to the other agent</message>
</communication>
<plan>
  <action>action_name</action>
  <ingredients>[ingredient0, ingredient1, ingredient2]</ingredients>
  <target_location>(x, y)</target_location>
  <description>Human-readable description of your plan</ …

- **user**: Conversation History:
[agent_0 to agent_1]: "I see that Agent 1 has reached the destination. How can I assist you further?"
[agent_1 to agent_0]: "Hi! My observation shows that I am currently at position (1, 1) on a 5x4 grid. I have no visible objects here but could use some help with my current recipe."


- **user**: You are agent_0. The other agent is agent_1.

Your Observation:
Agent at position (3, 1), facing south
Grid size: 5x4 (full observability)
Holding: nothing
Current recipe requires: 3x ingredient0

Visible objects (5 total):
  - pot at (2, 0) - empty [distance: 2 steps, northwest from you]
  - ingredient0 pile at (0, 1) [distance: 3 steps, west from you]
  - ingredient0 pile at (4, 1) [distance: 1 steps, east from you]
  - plate pile at (1, 3) [distance: 4 steps, southwest from you]
  - goal (delivery location) at (3, 3) [distance: 2 steps, south from you]

Generate your communication message to agent_1 and your action plan:

#### Completion (raw)
```
<communication>
  <to_agent>agent_1</to_agent>
  <message>"I see that Agent 1 has reached the destination. How can I assist you further?"</message>
</communication>
<plan>
  <action>cook_dish</action>
  <ingredients>ingredient0, ingredient1</ingredients>
  <target_location>(3, 3)</target_location>
  <description>We need to find a way to get to the delivery location without using any ingredients we have already cooked.</description>
</plan>
```

#### Conversation history

- [agent_0 → agent_1]: "I see that Agent 1 has reached the destination. How can I assist you further?"

- [agent_1 → agent_0]: "Hi! My observation shows that I am currently at position (1, 1) on a 5x4 grid. I have no visible objects here but could use some help with my current recipe."

- [system → agent_0]: Your plan was invalid: cook_dish should specify ingredients (executor can find pot automatically). Please provide a valid plan with all required fields in your next response.

- [system → agent_1]: Your plan was invalid: cook_dish should specify ingredients (executor can find pot automatically). Please provide a valid plan with all required fields in your next response.


### Sample 2 — `agent_1` | episode 0 step 0 | reward **175.63**

#### Prompt (chat messages)

- **system**: You are an agent in a cooperative cooking game called Overcooked. You work with another agent to cook dishes and deliver them to the goal.

Your task:
1. Observe your local environment
2. Communicate with the other agent about your plans and needs
3. Generate an action plan for what you will do next

Available actions:
- cook_dish: Cook a dish in a pot with specific ingredients
- pickup_ingredient: Pick up an ingredient from a pile
- pickup_plate: Pick up a plate from a plate pile
- deliver_dish: Deliver a completed dish to the goal
- wait: Wait for the other agent or for cooking to complete
- move_to: Move to a specific location

Respond in the following XML format:
<communication>
  <to_agent>agent_X</to_agent>
  <message>Your message to the other agent</message>
</communication>
<plan>
  <action>action_name</action>
  <ingredients>[ingredient0, ingredient1, ingredient2]</ingredients>
  <target_location>(x, y)</target_location>
  <description>Human-readable description of your plan</ …

- **user**: Conversation History:
[agent_0 to agent_1]: "I see that Agent 1 has reached the destination. How can I assist you further?"
[agent_1 to agent_0]: "Hi! My observation shows that I am currently at position (1, 1) on a 5x4 grid. I have no visible objects here but could use some help with my current recipe."
⚠️ SYSTEM: Your plan was invalid: cook_dish should specify ingredients (executor can find pot automatically). Please provide a valid plan with all required fields in your next response.


- **user**: You are agent_1. The other agent is agent_0.

Your Observation:
Agent at position (1, 1), facing southeast
Grid size: 5x4 (full observability)
Holding: nothing
Current recipe requires: 3x ingredient0

Visible objects (5 total):
  - pot at (2, 0) - empty [distance: 2 steps, northeast from you]
  - ingredient0 pile at (0, 1) [distance: 1 steps, west from you]
  - ingredient0 pile at (4, 1) [distance: 3 steps, east from you]
  - plate pile at (1, 3) [distance: 2 steps, south from you]
  - goal (delivery location) at (3, 3) [distance: 4 steps, southeast from you]

Generate your communication message to agent_0 and your action plan:

#### Completion (raw)
```
<communication>
  <to_agent>agent_0</to_agent>
  <message>"Hi! My observation shows that I am currently at position (1, 1) on a 5x4 grid. I have no visible objects here but could use some help with my current recipe."</message>
</communication>
<plan>
  <action>cook_dish</action>
  <ingredients>ingredient0</ingredients>
  <target_location>(3, 3)</target_location>
  <description>We need to cook our current dish first before moving on to the next one.</description>
</plan>
```

#### Conversation history

- [agent_0 → agent_1]: "I see that Agent 1 has reached the destination. How can I assist you further?"

- [agent_1 → agent_0]: "Hi! My observation shows that I am currently at position (1, 1) on a 5x4 grid. I have no visible objects here but could use some help with my current recipe."

- [system → agent_0]: Your plan was invalid: cook_dish should specify ingredients (executor can find pot automatically). Please provide a valid plan with all required fields in your next response.

- [system → agent_1]: Your plan was invalid: cook_dish should specify ingredients (executor can find pot automatically). Please provide a valid plan with all required fields in your next response.


### Sample 3 — `agent_0` | episode 0 step 1 | reward **175.63**

#### Prompt (chat messages)

- **system**: You are an agent in a cooperative cooking game called Overcooked. You work with another agent to cook dishes and deliver them to the goal.

Your task:
1. Observe your local environment
2. Communicate with the other agent about your plans and needs
3. Generate an action plan for what you will do next

Available actions:
- cook_dish: Cook a dish in a pot with specific ingredients
- pickup_ingredient: Pick up an ingredient from a pile
- pickup_plate: Pick up a plate from a plate pile
- deliver_dish: Deliver a completed dish to the goal
- wait: Wait for the other agent or for cooking to complete
- move_to: Move to a specific location

Respond in the following XML format:
<communication>
  <to_agent>agent_X</to_agent>
  <message>Your message to the other agent</message>
</communication>
<plan>
  <action>action_name</action>
  <ingredients>[ingredient0, ingredient1, ingredient2]</ingredients>
  <target_location>(x, y)</target_location>
  <description>Human-readable description of your plan</ …

- **user**: Conversation History:
[agent_0 to agent_1]: "I see that Agent 1 has reached the destination. How can I assist you further?"
[agent_1 to agent_0]: "Hi! My observation shows that I am currently at position (1, 1) on a 5x4 grid. I have no visible objects here but could use some help with my current recipe."
⚠️ SYSTEM: Your plan was invalid: cook_dish should specify ingredients (executor can find pot automatically). Please provide a valid plan with all required fields in your next response.
⚠️ SYSTEM: Your plan was invalid: cook_dish should specify ingredients (executor can find pot automatically). Please provide a valid plan with all required fields in your next response.


- **user**: You are agent_0. The other agent is agent_1.

Your Observation:
Agent at position (3, 1), facing south
Grid size: 5x4 (full observability)
Holding: nothing
Current recipe requires: 3x ingredient0

Visible objects (5 total):
  - pot at (2, 0) - empty [distance: 2 steps, northwest from you]
  - ingredient0 pile at (0, 1) [distance: 3 steps, west from you]
  - ingredient0 pile at (4, 1) [distance: 1 steps, east from you]
  - plate pile at (1, 3) [distance: 4 steps, southwest from you]
  - goal (delivery location) at (3, 3) [distance: 2 steps, south from you]

Generate your communication message to agent_1 and your action plan:

#### Completion (raw)
```
<communication>
  <to_agent>agent_1</to_agent>
  <message>Agent 1, we need to find a pot to start cooking our dish.</message>
</communication>

<plan>
  <action>wait</action>
  <ingredients>[]</ingredients>
  <target_location>(0, 0)</target_location>
  <description>You need to find a pot to start cooking your dish.</description>
</plan>
```

#### Conversation history

- [agent_1 → agent_0]: "Hi! My observation shows that I am currently at position (1, 1) on a 5x4 grid. I have no visible objects here but could use some help with my current recipe."

- [system → agent_0]: Your plan was invalid: cook_dish should specify ingredients (executor can find pot automatically). Please provide a valid plan with all required fields in your next response.

- [system → agent_1]: Your plan was invalid: cook_dish should specify ingredients (executor can find pot automatically). Please provide a valid plan with all required fields in your next response.

- [agent_0 → agent_1]: Agent 1, we need to find a pot to start cooking our dish.

- [agent_1 → agent_0]: Your message to agent_0:


In [30]:
# Inspect a specific index (change INDEX to any 0 .. len(ds)-1)
INDEX = 301
show_sample(ds[INDEX], INDEX + 1)

### Sample 302 — `agent_1` | episode 1 step 0 | reward **180.64**

#### Prompt (chat messages)

- **system**: You are an agent in a cooperative cooking game called Overcooked. You work with another agent to cook dishes and deliver them to the goal.

Your task:
1. Observe your local environment
2. Communicate with the other agent about your plans and needs
3. Generate an action plan for what you will do next

Available actions:
- cook_dish: Cook a dish in a pot with specific ingredients
- pickup_ingredient: Pick up an ingredient from a pile
- pickup_plate: Pick up a plate from a plate pile
- deliver_dish: Deliver a completed dish to the goal
- wait: Wait for the other agent or for cooking to complete
- move_to: Move to a specific location

Respond in the following XML format:
<communication>
  <to_agent>agent_X</to_agent>
  <message>Your message to the other agent</message>
</communication>
<plan>
  <action>action_name</action>
  <ingredients>[ingredient0, ingredient1, ingredient2]</ingredients>
  <target_location>(x, y)</target_location>
  <description>Human-readable description of your plan</ …

- **user**: Conversation History:
[agent_0 to agent_1]: You should start by moving to the north side of the grid.
[agent_1 to agent_0]: You need to cook some food to prepare your meal. Please follow my instructions.
⚠️ SYSTEM: Your plan was invalid: Invalid action: move_to(1, 3). Please provide a valid plan with all required fields in your next response.


- **user**: You are agent_1. The other agent is agent_0.

Your Observation:
Agent at position (1, 1), facing southeast
Grid size: 5x4 (full observability)
Holding: nothing
Current recipe requires: 3x ingredient0

Visible objects (5 total):
  - pot at (2, 0) - empty [distance: 2 steps, northeast from you]
  - ingredient0 pile at (0, 1) [distance: 1 steps, west from you]
  - ingredient0 pile at (4, 1) [distance: 3 steps, east from you]
  - plate pile at (1, 3) [distance: 2 steps, south from you]
  - goal (delivery location) at (3, 3) [distance: 4 steps, southeast from you]

Generate your communication message to agent_0 and your action plan:

#### Completion (raw)
```
<communication>
  <to_agent>agent_0</to_agent>
  <message>You need to cook some food to prepare your meal. Please follow my instructions.</message>
</communication>
<plan>
  <action>cook_dish</action>
  <ingredients>[ingredient0, ingredient0, ingredient0]</ingredients>
  <target_location>(3, 3)</target_location>
  <description>We will cook three pieces of food for you to eat. This will be the main course of your dinner tonight.</description>
</plan>
```

#### Conversation history

- [agent_0 → agent_1]: You should start by moving to the north side of the grid.

- [agent_1 → agent_0]: You need to cook some food to prepare your meal. Please follow my instructions.

- [system → agent_0]: Your plan was invalid: Invalid action: move_to(1, 3). Please provide a valid plan with all required fields in your next response.

- [system → agent_1]: Your plan was invalid: cook_dish should specify ingredients (executor can find pot automatically). Please provide a valid plan with all required fields in your next response.


## 3. How GRPOTrainer processes the data

This section shows how the raw dataset is transformed into the format that GRPOTrainer (and the model) actually receives:
1. **Raw dataset entry** (what we saw above)
2. **Tokenized format** (how the tokenizer converts prompt+completion)
3. **Model input tensors** (input_ids, attention_mask, labels, etc.)

In [31]:
# Load the same tokenizer used in training (adjust model_name if different)
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # Change if you used a different model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Same as in training

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

Tokenizer: Qwen/Qwen2.5-0.5B-Instruct
Vocab size: 151665
Pad token: <|endoftext|> (ID: 151643)
EOS token: <|im_end|> (ID: 151645)


In [32]:
def show_grpo_processing(ex, tokenizer, idx: int = 0):
    """Show how GRPOTrainer processes one example."""
    from IPython.display import display, Markdown, Code
    
    md = [f"## GRPOTrainer Processing — Sample {idx + 1}\n"]
    
    # 1. Raw dataset entry
    md.append("### 1. Raw Dataset Entry\n")
    md.append(f"- **prompt**: {len(ex['prompt'])} chat messages\n")
    md.append(f"- **completion**: {len(ex['completion'])} chars\n")
    md.append(f"- **reward**: {ex['reward']}\n")
    
    # 2. Apply chat template (convert messages to text)
    md.append("\n### 2. Chat Template Application\n")
    md.append("GRPOTrainer applies the tokenizer's chat template to convert messages to text:\n\n")
    
    # Apply chat template
    try:
        # For Qwen, apply_chat_template expects messages in the format we have
        prompt_text = tokenizer.apply_chat_template(
            ex["prompt"], 
            tokenize=False, 
            add_generation_prompt=True  # Adds assistant prompt
        )
        md.append("**Full prompt text (after chat template):**\n")
        md.append(f"```\n{prompt_text}\n```\n")
    except Exception as e:
        md.append(f"*Error applying chat template: {e}*\n")
        # Fallback: manually format
        prompt_text = ""
        for m in ex["prompt"]:
            role = m.get("role", "?")
            content = m.get("content", m.get("text", ""))
            prompt_text += f"{role}: {content}\n\n"
        md.append("**Manual format (fallback):**\n")
        md.append(f"```\n{prompt_text}\n```\n")
    
    # 3. Tokenize prompt + completion
    md.append("\n### 3. Tokenization (Prompt + Completion)\n")
    full_text = prompt_text + ex["completion"]
    
    # Tokenize
    tokenized = tokenizer(
        full_text,
        return_tensors="pt",
        padding=False,
        truncation=False,
        add_special_tokens=True
    )
    
    input_ids = tokenized["input_ids"][0].tolist()
    attention_mask = tokenized["attention_mask"][0].tolist()
    
    # Find where prompt ends and completion starts
    prompt_tokenized = tokenizer(
        prompt_text,
        return_tensors="pt",
        padding=False,
        truncation=False,
        add_special_tokens=True
    )
    prompt_len = len(prompt_tokenized["input_ids"][0])
    
    md.append(f"- **Total tokens**: {len(input_ids)}\n")
    md.append(f"- **Prompt tokens**: {prompt_len}\n")
    md.append(f"- **Completion tokens**: {len(input_ids) - prompt_len}\n")
    
    # Show token IDs
    md.append("\n**Token IDs (first 50 tokens):**\n")
    md.append(f"```\n{input_ids[:50]}\n```\n")
    
    # Decode to show what model sees
    md.append("\n**Decoded tokens (first 50):**\n")
    decoded = tokenizer.decode(input_ids[:50], skip_special_tokens=False)
    md.append(f"```\n{decoded}\n```\n")
    
    # 4. Model input format
    md.append("\n### 4. Model Input Format (Batch)\n")
    md.append("When batched, GRPOTrainer creates:\n\n")
    md.append("- **input_ids**: `[batch_size, seq_len]` - Token IDs\n")
    md.append("- **attention_mask**: `[batch_size, seq_len]` - 1 for real tokens, 0 for padding\n")
    md.append("- **labels**: `[batch_size, seq_len]` - Same as input_ids, but -100 for prompt tokens (only completion is trained)\n")
    
    # Show example with padding (simulate batching)
    md.append("\n**Example with padding (seq_len=100, this example padded to 100):**\n")
    padded_ids = input_ids[:100] + [tokenizer.pad_token_id] * max(0, 100 - len(input_ids))
    padded_mask = attention_mask[:100] + [0] * max(0, 100 - len(attention_mask))
    md.append(f"- input_ids shape: `[1, 100]`\n")
    md.append(f"- attention_mask shape: `[1, 100]`\n")
    md.append(f"- First 20 input_ids: `{padded_ids[:20]}`\n")
    md.append(f"- First 20 attention_mask: `{padded_mask[:20]}`\n")
    
    # Show labels (prompt masked with -100)
    labels = [-100] * prompt_len + input_ids[prompt_len:]
    if len(labels) < 100:
        labels = labels + [-100] * (100 - len(labels))
    md.append(f"- labels (first 20, -100 = ignored in loss): `{labels[:20]}`\n")
    md.append(f"  → Only completion tokens (after position {prompt_len}) contribute to loss\n")
    
    display(Markdown("\n".join(md)))

# Show processing for first example
show_grpo_processing(ds[0], tokenizer, idx=0)

## GRPOTrainer Processing — Sample 1

### 1. Raw Dataset Entry

- **prompt**: 3 chat messages

- **completion**: 443 chars

- **reward**: 175.63250732421875


### 2. Chat Template Application

GRPOTrainer applies the tokenizer's chat template to convert messages to text:


**Full prompt text (after chat template):**

```
<|im_start|>system
You are an agent in a cooperative cooking game called Overcooked. You work with another agent to cook dishes and deliver them to the goal.

Your task:
1. Observe your local environment
2. Communicate with the other agent about your plans and needs
3. Generate an action plan for what you will do next

Available actions:
- cook_dish: Cook a dish in a pot with specific ingredients
- pickup_ingredient: Pick up an ingredient from a pile
- pickup_plate: Pick up a plate from a plate pile
- deliver_dish: Deliver a completed dish to the goal
- wait: Wait for the other agent or for cooking to complete
- move_to: Move to a specific location

Respond in the following XML format:
<communication>
  <to_agent>agent_X</to_agent>
  <message>Your message to the other agent</message>
</communication>
<plan>
  <action>action_name</action>
  <ingredients>[ingredient0, ingredient1, ingredient2]</ingredients>
  <target_location>(x, y)</target_location>
  <description>Human-readable description of your plan</description>
</plan><|im_end|>
<|im_start|>user
Conversation History:
[agent_0 to agent_1]: "I see that Agent 1 has reached the destination. How can I assist you further?"
[agent_1 to agent_0]: "Hi! My observation shows that I am currently at position (1, 1) on a 5x4 grid. I have no visible objects here but could use some help with my current recipe."
<|im_end|>
<|im_start|>user
You are agent_0. The other agent is agent_1.

Your Observation:
Agent at position (3, 1), facing south
Grid size: 5x4 (full observability)
Holding: nothing
Current recipe requires: 3x ingredient0

Visible objects (5 total):
  - pot at (2, 0) - empty [distance: 2 steps, northwest from you]
  - ingredient0 pile at (0, 1) [distance: 3 steps, west from you]
  - ingredient0 pile at (4, 1) [distance: 1 steps, east from you]
  - plate pile at (1, 3) [distance: 4 steps, southwest from you]
  - goal (delivery location) at (3, 3) [distance: 2 steps, south from you]

Generate your communication message to agent_1 and your action plan:<|im_end|>
<|im_start|>assistant

```


### 3. Tokenization (Prompt + Completion)

- **Total tokens**: 639

- **Prompt tokens**: 529

- **Completion tokens**: 110


**Token IDs (first 50 tokens):**

```
[151644, 8948, 198, 2610, 525, 458, 8315, 304, 264, 47466, 17233, 1809, 2598, 6065, 98697, 13, 1446, 975, 448, 2441, 8315, 311, 4296, 25779, 323, 6359, 1105, 311, 279, 5795, 382, 7771, 3383, 510, 16, 13, 4657, 13267, 697, 2205, 4573, 198, 17, 13, 16433, 349, 448, 279, 1008, 8315]
```


**Decoded tokens (first 50):**

```
<|im_start|>system
You are an agent in a cooperative cooking game called Overcooked. You work with another agent to cook dishes and deliver them to the goal.

Your task:
1. Observe your local environment
2. Communicate with the other agent
```


### 4. Model Input Format (Batch)

When batched, GRPOTrainer creates:


- **input_ids**: `[batch_size, seq_len]` - Token IDs

- **attention_mask**: `[batch_size, seq_len]` - 1 for real tokens, 0 for padding

- **labels**: `[batch_size, seq_len]` - Same as input_ids, but -100 for prompt tokens (only completion is trained)


**Example with padding (seq_len=100, this example padded to 100):**

- input_ids shape: `[1, 100]`

- attention_mask shape: `[1, 100]`

- First 20 input_ids: `[151644, 8948, 198, 2610, 525, 458, 8315, 304, 264, 47466, 17233, 1809, 2598, 6065, 98697, 13, 1446, 975, 448, 2441]`

- First 20 attention_mask: `[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]`

- labels (first 20, -100 = ignored in loss): `[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]`

  → Only completion tokens (after position 529) contribute to loss


### Summary: Data Flow

1. **Dataset** → `prompt` (list of chat messages) + `completion` (string)
2. **Chat Template** → Converts messages to a single text string with special tokens
3. **Tokenization** → Converts text to token IDs (`input_ids`)
4. **Batching** → Pads sequences to same length, creates `attention_mask`
5. **Labels** → Same as `input_ids`, but prompt tokens set to `-100` (ignored in loss)
6. **Model Forward** → Receives `input_ids`, `attention_mask`; predicts next tokens
7. **Loss** → Only computed on completion tokens (where `labels != -100`)

**Key point**: The model only learns to predict the completion (the XML response), not the prompt!

In [18]:
# Quick comparison: Raw vs Tokenized for a specific example
EXAMPLE_IDX = 0
ex = ds[EXAMPLE_IDX]

print("=" * 80)
print("RAW DATASET FORMAT")
print("=" * 80)
print(f"prompt (type: {type(ex['prompt'])}):")
for i, msg in enumerate(ex['prompt']):
    print(f"  [{i}] {msg.get('role')}: {msg.get('content', msg.get('text', ''))[:100]}...")
print(f"\ncompletion (type: {type(ex['completion'])}):")
print(f"  {ex['completion'][:200]}...")

print("\n" + "=" * 80)
print("TOKENIZED FORMAT (what model receives)")
print("=" * 80)

# Apply chat template
prompt_text = tokenizer.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
full_text = prompt_text + ex["completion"]
tokenized = tokenizer(full_text, return_tensors="pt", padding=False)

print(f"input_ids (shape: {tokenized['input_ids'].shape}):")
print(f"  {tokenized['input_ids'][0][:30].tolist()}...")
print(f"\nDecoded back to text (first 200 chars):")
print(f"  {tokenizer.decode(tokenized['input_ids'][0][:30], skip_special_tokens=False)}...")
print(f"\nTotal sequence length: {tokenized['input_ids'].shape[1]} tokens")

RAW DATASET FORMAT
prompt (type: <class 'list'>):
  [0] system: You are an agent in a cooperative cooking game called Overcooked. You work with another agent to coo...
  [1] user: You are agent_0. The other agent is agent_1.

Your Observation:
Agent at position (3, 1), facing sou...

completion (type: <class 'str'>):
  <communication>
  <to_agent>agent_1</to_agent>
  <message>"I see Agent 1 holding no ingredients. Let's get started with the recipe.">
</communication>

<plan>
  <action>pickup_ingredient</action>
  <i...

TOKENIZED FORMAT (what model receives)
input_ids (shape: torch.Size([1, 537])):
  [151644, 8948, 198, 2610, 525, 458, 8315, 304, 264, 47466, 17233, 1809, 2598, 6065, 98697, 13, 1446, 975, 448, 2441, 8315, 311, 4296, 25779, 323, 6359, 1105, 311, 279, 5795]...

Decoded back to text (first 200 chars):
  <|im_start|>system
You are an agent in a cooperative cooking game called Overcooked. You work with another agent to cook dishes and deliver them to the goal...

Total seque